In [ ]:
!pip3 install kafka-python-ng
!pip install mysql-connector-python
! pip3 install s3fs
!pip install boto3



In [ ]:
from kafka import KafkaConsumer
import mysql.connector
import json
import csv
import io
import boto3
from datetime import datetime

# === AWS S3 Configuration ===
AWS_ACCESS_KEY = ''
AWS_SECRET_KEY = ''
S3_BUCKET_NAME = 'ashishbucket08'
S3_FOLDER = 'zomato-data/'

# Boto3 client setup
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='ap-south-1'
)

# === Kafka Consumer Setup ===
consumer = KafkaConsumer(
    'zomato-topic',
    bootstrap_servers='3.110.108.178:9092',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    group_id='zomato-group',
    auto_offset_reset='earliest',
    enable_auto_commit=True
)

# === MySQL Setup ===
connection = mysql.connector.connect(
    host='zomato-db.cniui0ok0s37.ap-south-1.rds.amazonaws.com',
    user='admin',
    password='MySecurePassword123',
    database='zomato_db'
)
cursor = connection.cursor()

columns_list = [
    'url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'phone',
    'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost',
    'reviews_list', 'menu_item', 'listed_in_type', 'listed_in_city'
]

insert_query = f"""
INSERT INTO zomato_table ({', '.join(columns_list)})
VALUES ({', '.join(['%s'] * len(columns_list))})
"""

# === Consume and Process ===
for message in consumer:
    raw_csv_line = message.value.get('data', '')
    if not raw_csv_line:
        print("⚠️ Empty message received, skipping.")
        continue

    reader = csv.reader(io.StringIO(raw_csv_line))
    try:
        parsed_columns = next(reader)
    except Exception as e:
        print(f"⚠️ Error parsing row: {e}")
        continue

    if len(parsed_columns) == len(columns_list):
        # Clean/normalize values
        cleaned_columns = [col if col.strip() not in ['[]', '', 'null'] else None for col in parsed_columns]

        # Create quoted CSV buffer for S3
        csv_buffer = io.StringIO()
        writer = csv.writer(csv_buffer, quoting=csv.QUOTE_ALL)
        writer.writerow(columns_list)
        writer.writerow(cleaned_columns)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        s3_key = f"{S3_FOLDER}zomato_record_{timestamp}.csv"

        # Upload to S3
        try:
            s3_client.put_object(
                Bucket=S3_BUCKET_NAME,
                Key=s3_key,
                Body=csv_buffer.getvalue()
            )
            print(f"☁️ Uploaded to S3: {s3_key}")
        except Exception as e:
            print(f"❌ S3 Upload failed: {e}")

        # Insert into MySQL
        try:
            cursor.execute(insert_query, cleaned_columns)
            connection.commit()
            print("✅ Inserted into DB:", cleaned_columns[:3])
        except Exception as e:
            print(f"❌ MySQL insert error: {e}")
    else:
        print(f"⚠️ Column count mismatch: Expected {len(columns_list)}, Got {len(parsed_columns)}. Row skipped.")

# Cleanup
cursor.close()
connection.close()
